In [4]:
!pip install langchain-tavily ddgs

  Using cached ddgs-9.11.2-py3-none-any.whl.metadata (14 kB)
Using cached ddgs-9.11.2-py3-none-any.whl (43 kB)


In [5]:
from typing import Annotated, Sequence, TypedDict, Dict, List, Optional 
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI 
from langchain_core.messages import BaseMessage, ToolMessage, SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool 
from langgraph.graph.message import add_messages # reducer
from langgraph.graph import StateGraph, START, END 
from langgraph.prebuilt import ToolNode # Create tools 
from IPython.display import display, Image
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import PyMuPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.tools import DuckDuckGoSearchRun # Searching 
import os 
# Api stuff
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware 
from fastapi.responses import StreamingResponse 
from pydantic import BaseModel 
import json 

load_dotenv()

search_tool = DuckDuckGoSearchRun(max_results=3)
tools = [search_tool]

llm = ChatOpenAI(
    model = "stepfun/step-3.5-flash:free",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTERKEY"]).bind_tools(tools)

embeddings = OpenAIEmbeddings(
    model = "openai/text-embedding-3-small",
    openai_api_base="https://openrouter.ai/api/v1",
    openai_api_key=os.environ["OPENROUTERKEY"]
)

In [10]:
!pip install duckduckgo-search

  Using cached duckduckgo_search-8.1.1-py3-none-any.whl.metadata (16 kB)
  Using cached primp-1.1.2-cp310-abi3-macosx_11_0_arm64.whl.metadata (6.7 kB)
  Using cached lxml-6.0.2-cp313-cp313-macosx_10_13_universal2.whl.metadata (3.6 kB)
Using cached duckduckgo_search-8.1.1-py3-none-any.whl (18 kB)
Using cached lxml-6.0.2-cp313-cp313-macosx_10_13_universal2.whl (8.6 MB)
Using cached primp-1.1.2-cp310-abi3-macosx_11_0_arm64.whl (3.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [duckduckgo-search]duckduckgo-search]


In [ ]:
import time
import random
from duckduckgo_search import DDGS
from langgraph.prebuilt import create_react_agent
from langchain_openai import ChatOpenAI

# The "Human-Like" Search Tool
def safe_web_search(query: str):
    """Searches the web for free without getting banned."""
    # Tiny random sleep so you don't look like a rapid-fire bot
    time.sleep(random.uniform(1, 2)) 
    
    results = []
    with DDGS() as ddgs:
        # 'max_results=3' keeps the request small and safe
        for r in ddgs.text(query, max_results=3):
            results.append(f"Title: {r['title']}\nSnippet: {r['body']}\nSource: {r['href']}")
    print(results)
    return "\n\n".join(results)

# Setup Agent
agent = llm.bind_tools([safe_web_search])

# Does invoke it, add graph to make this work now
response = agent.invoke("What is the latest news about SpaceX?")
print(response)


content="I'll search for the latest news about SpaceX for you." additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 99, 'prompt_tokens': 237, 'total_tokens': 336, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 63, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0}}, 'model_provider': 'openai', 'model_name': 'stepfun/step-3.5-flash:free', 'system_fingerprint': None, 'id': 'gen-1772938523-XPvdlvkUmXepa9FcBoTS', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019ccb5e-ef07-7453-b821-a3098bde1a97-0' tool_calls=[{'name': 'safe_web_search', 'args': {'query': 'SpaceX latest news 2024 recent updates'}, 'id': 'call_f3674

In [19]:
from pprint import pprint

response.content

"I'll search for the latest news about SpaceX for you."